In [ ]:
!sudo apt-get install zstd
!pip install fastapi uvicorn pyngrok nest-asyncio
!pip install diffusers transformers accelerate
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,842 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently

In [ ]:
import subprocess
import time

# Inicia o Ollama em background
print("Iniciando o servidor Ollama...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Dá um tempo para o servidor subir

# Baixa o modelo desejado (isso pode levar alguns minutos na primeira vez)
print("Baixando o modelo Llama 3...")
subprocess.run(["ollama", "pull", "llama3"])
print("Ollama pronto!")

Iniciando o servidor Ollama...
Baixando o modelo Llama 3...
Ollama pronto!


## Rodar o modelo

In [ ]:
import torch
import base64
import json
import subprocess
from io import BytesIO
from PIL import Image
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
import httpx

# ==========================================
# ⚙️ CHAVE DE HARDWARE (Mude aqui!)
# ==========================================
USAR_GPU = True  # Coloque False se for rodar no Colab sem placa de vídeo
# ==========================================

app = FastAPI(title="AI Colab Backend")

if USAR_GPU:
    print("Carregando Stable Diffusion (Text-to-Image) na GPU 🚀...")
    pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
    pipe = pipe.to("cuda")
else:
    print("Carregando Stable Diffusion (Text-to-Image) na CPU 🐢...")
    pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")

print("Carregando Stable Diffusion (Image-to-Image)...")
pipe_i2i = StableDiffusionImg2ImgPipeline(**pipe.components)

print("Modelos carregados com sucesso!")

# --- SCHEMAS ---
class TextRequest(BaseModel):
    prompt: str
    modelo: str = "llama3"
    contexto_sistema: str = ""
    stream: bool = True
    imagem_base64: str = None

class ImageRequest(BaseModel):
    prompt: str
    imagem_base64: str = None

class PullRequest(BaseModel):
    name: str

# --- ROTAS DE TEXTO (OLLAMA) ---
@app.post("/gerar_texto")
async def gerar_texto(req: TextRequest):
    url_ollama = "http://localhost:11434/api/generate"

    payload = {
        "model": req.modelo,
        "prompt": req.prompt,
        "system": req.contexto_sistema,
        "stream": req.stream
    }

    if req.imagem_base64:
        print("📸 UHUU! O COLAB RECEBEU A IMAGEM PARA LER!")
        img_str = req.imagem_base64.split(",")[1] if "," in req.imagem_base64 else req.imagem_base64
        payload["images"] = [img_str]
    else:
        print("📝 Nenhuma imagem recebida nesta requisição.")

    if req.stream:
        async def gerador_stream():
            async with httpx.AsyncClient(timeout=120.0) as client:
                async with client.stream("POST", url_ollama, json=payload) as resposta:
                    async for linha in resposta.aiter_lines():
                        if linha:
                            yield linha + "\n"
        return StreamingResponse(gerador_stream(), media_type="application/x-ndjson")
    else:
        async with httpx.AsyncClient(timeout=120.0) as client:
            resposta = await client.post(url_ollama, json=payload)
        if resposta.status_code != 200:
            raise HTTPException(status_code=500, detail="Erro no Ollama")
        return {"resposta": resposta.json()["response"]}

# --- ROTAS DE IMAGEM (STABLE DIFFUSION) ---
@app.post("/gerar_imagem")
def gerar_imagem(req: ImageRequest):
    try:
        if req.imagem_base64:
            print("Gerando imagem via Img2Img...")
            img_str = req.imagem_base64.split(",")[1] if "," in req.imagem_base64 else req.imagem_base64
            img_data = base64.b64decode(img_str)
            init_image = Image.open(BytesIO(img_data)).convert("RGB")
            init_image = init_image.resize((512, 512))
            image = pipe_i2i(prompt=req.prompt, image=init_image, strength=0.75, guidance_scale=7.5).images[0]
        else:
            print("Gerando imagem via Txt2Img...")
            image = pipe(req.prompt).images[0]

        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_str_out = base64.b64encode(buffered.getvalue()).decode("utf-8")
        return {"imagem_base64": img_str_out}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- ROTAS DO GESTOR DE MODELOS ---
@app.get("/tags")
async def listar_tags():
    async with httpx.AsyncClient() as client:
        resp = await client.get("http://localhost:11434/api/tags")
        return resp.json()

def baixar_no_background(nome_modelo: str):
    print(f"\n⏳ INICIANDO DOWNLOAD DO MODELO: {nome_modelo} (Isso vai demorar...)")
    try:
        import httpx
        with httpx.Client(timeout=None) as client:
            resp = client.post("http://localhost:11434/api/pull", json={"name": nome_modelo, "stream": False})
            if resp.status_code == 200:
                print(f"✅ DOWNLOAD CONCLUÍDO COM SUCESSO: {nome_modelo}")
            else:
                print(f"❌ O OLLAMA RECUSOU O DOWNLOAD DE {nome_modelo}: {resp.text}")
    except Exception as e:
        print(f"❌ ERRO CRÍTICO DE REDE INTERNA NO COLAB: {e}")

@app.post("/pull")
async def puxar_modelo(req: PullRequest, bg_tasks: BackgroundTasks):
    print(f"\n📥 Ordem de download recebida da sua interface para: {req.name}")
    bg_tasks.add_task(baixar_no_background, req.name)
    return {"status": "Download delegado para o FastAPI com sucesso!"}

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Carregando Stable Diffusion (Text-to-Image) na GPU 🚀...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Carregando Stable Diffusion (Image-to-Image)...
Modelos carregados com sucesso!


## Iniciar túnel

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

# Limpa túneis antigos caso você rode a célula mais de uma vez
ngrok.kill()

# Coloque o seu token do Ngrok aqui novamente
NGROK_TOKEN = "3AdnGzwcXtwO7t0eFODcqas40yN_7msYwirg5kcEhfu9LNBAB"
ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
print("=" * 50)
print(f"URL PÚBLICA DO SEU COLAB: {tunnel.public_url}")
print("Copie essa URL, vamos usá-la no seu servidor local!")
print("=" * 50)

def run_server():
    # Executa o servidor sem causar conflito de loop no Colab
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Inicia o servidor silenciosamente em segundo plano
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

URL PÚBLICA DO SEU COLAB: https://unaddled-raymundo-unbowed.ngrok-free.dev
Copie essa URL, vamos usá-la no seu servidor local!


In [ ]:
!ollama list

NAME             ID              SIZE      MODIFIED      
llava:latest     8dd30f6b0cb1    4.7 GB    3 seconds ago    
llama3:latest    365c0bd3c000    4.7 GB    3 minutes ago    


No journal files were found.
-- No entries --
>